# Дообучение YOLO: MOCS + доп. датасеты техники

Продолжаем обучение модели, уже прошедшей 50 эпох на MOCS, на дополнительных датасетах
(kaggle `xyzyxzzxy/construction-equipment`, `kartaviychert/arh-df`, `dataclusterlabs/construction-vehicle-images`,
roboflow `jejung/construction-machinery-tbosw`).

Принципы, заложенные в ноутбук:
- **Никакого дублирования данных на диске.** Изображения не копируются, а симлинкуются
  (`link_or_copy`) — особенно важно для MOCS (41 668 изображений).
- **Датасеты не сливаются физически.** У каждого — своя папка `*_yolo/`, а в общий `data.yaml`
  они попадают просто как несколько путей в списке `train:`. YOLO/Ultralytics поддерживает это нативно.
- **Классы новых датасетов ремаппятся** в ваши канонические 13 классов (MOCS-таксономия) по имени,
  а не по числовому индексу — числовые id в разных источниках не совпадают.
- **Дообучение — не `resume`.** Веса грузятся как стартовая точка, обучение идёт в две фазы:
  сначала с замороженным backbone (несколько эпох, низкий риск "смыть" уже выученное), потом полностью
  разморожено с низким learning rate.
- **`cache=False` в `model.train()`** — не кэшируем resized-картинки на диск/в RAM, датасет с MOCS внутри слишком большой.

## 0. Установка зависимостей

In [ ]:
!pip install ultralytics opencv-python pandas pycocotools pyyaml -q

## 1. Конфигурация

Заполните здесь пути:
- `PREV_BEST_WEIGHTS` — путь к весам после ваших 50 эпох на MOCS (`best.pt`).
- `MOCS_YOLO_DIR` — папка, куда (или где уже) сконвертирован MOCS в YOLO-формат.
  Если она уже существует с прошлого запуска — конвертацию MOCS заново запускать не нужно
  (см. ячейку в разделе 2, она сама это проверяет и пропускает шаг).

In [ ]:
import os
from pathlib import Path

RUNNING_ON_KAGGLE = os.path.exists("/kaggle/input")

# --- пути, которые нужно проверить/поправить под свою среду ---
PREV_BEST_WEIGHTS = r"C:\Users\edwar\PycharmProjects\JupyterProject\cv\runs\detect\runs_mocs\train\weights\best.pt"   # веса после 50 эпох на MOCS
MOCS_YOLO_DIR = Path("mocs_yolo")                        # уже сконвертированный MOCS (если есть)
WORK_ROOT = Path("finetune_data")                         # сюда лягут только НОВЫЕ (маленькие) датасеты
WORK_ROOT.mkdir(exist_ok=True)

# Канонические 13 классов — порядок ДОЛЖЕН совпадать с тем, что было при обучении на MOCS
CANONICAL_CLASSES = [
    "worker", "tower_crane", "hanging_hook", "vehicle_crane", "roller", "bulldozer",
    "excavator", "truck", "loader", "pump_truck", "concrete_mixer", "pile_driver", "other_vehicle",
]
CANON_IDX = {name: i for i, name in enumerate(CANONICAL_CLASSES)}
print("Канонические классы:", CANONICAL_CLASSES)
print("Запуск на Kaggle:", RUNNING_ON_KAGGLE)

## 2. Диск-дружелюбные хелперы

`link_or_copy` — ядро экономии места: символическая ссылка вместо копирования файла.
Если по каким-то причинам симлинки недоступны (редкость на Linux/Kaggle/Colab, актуально
в основном для Windows без прав администратора) — падаем обратно на обычное копирование.

In [ ]:
import json, shutil, random, tempfile
import xml.etree.ElementTree as ET
import yaml


_SYMLINK_SUPPORTED = None


def _check_symlink_support():
    """Проверяем ОДИН РАЗ, поддерживаются ли symlink-и в этой среде — на Windows без
    включённого Developer Mode/прав администратора os.symlink тихо падает, и без
    этой проверки вы бы молча копировали гигабайты файлов на каждый вызов
    link_or_copy, думая, что диск экономится, а на деле — нет."""
    global _SYMLINK_SUPPORTED
    if _SYMLINK_SUPPORTED is None:
        test_dir = Path(tempfile.gettempdir()) / "symlink_capability_test"
        test_dir.mkdir(exist_ok=True)
        src, dst = test_dir / "s.txt", test_dir / "d.txt"
        src.write_text("x")
        try:
            os.symlink(src, dst)
            _SYMLINK_SUPPORTED = True
        except OSError:
            _SYMLINK_SUPPORTED = False
        shutil.rmtree(test_dir, ignore_errors=True)
        if not _SYMLINK_SUPPORTED:
            print("!! Символические ссылки недоступны в этой среде (на Windows нужен "
                  "включённый Developer Mode или запуск от администратора). "
                  "Файлы будут КОПИРОВАТЬСЯ вместо симлинков — это займёт больше места "
                  "на диске и больше времени.", flush=True)
    return _SYMLINK_SUPPORTED


def link_or_copy(src_path, dst_path):
    """Симлинк вместо копирования — почти нулевой расход диска.
    Если symlink-и недоступны в этой среде — сразу копируем, не тратя время
    на повторную неудачную попытку симлинка на каждом из тысяч файлов."""
    dst_path = Path(dst_path)
    if dst_path.exists() or dst_path.is_symlink():
        return
    if _check_symlink_support():
        try:
            os.symlink(Path(src_path).resolve(), dst_path)
            return
        except OSError:
            pass
    shutil.copy(src_path, dst_path)


def _looks_like_coco(json_path):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return isinstance(data, dict) and {"images", "annotations", "categories"} <= set(data.keys())
    except Exception:
        return False


def _find_yolo_split_dirs(root):
    """Ищет пары images/+labels/ как соседей (train/, valid/, val/, test/) -
    типичная YOLO-раскладка, даже без data.yaml/classes.txt (частый случай
    на переупакованных kaggle-зеркалах roboflow-датасетов)."""
    split_dirs = []
    for images_dir in Path(root).rglob("images"):
        if images_dir.is_dir():
            labels_dir = images_dir.parent / "labels"
            if labels_dir.is_dir():
                split_dirs.append(images_dir.parent)
    return split_dirs


def detect_annotation_format(dataset_root):
    """Определяет формат разметки:
    'yolo' - есть data.yaml с полем names;
    'yolo_nolabelmap' - есть images/+labels/, но yaml/classes.txt нет
        (нужно будет передать class_names_override вручную);
    'coco' - json с images/annotations/categories;
    'voc' - xml."""
    root = Path(dataset_root)

    for yf in list(root.rglob("*.yaml")) + list(root.rglob("*.yml")):
        try:
            content = yaml.safe_load(open(yf, encoding="utf-8"))
            if isinstance(content, dict) and "names" in content:
                return "yolo", yf
        except Exception:
            continue

    json_files = [f for f in root.rglob("*.json") if _looks_like_coco(f)]
    if json_files:
        return "coco", json_files

    xml_files = list(root.rglob("*.xml"))
    if xml_files:
        return "voc", xml_files

    split_dirs = _find_yolo_split_dirs(root)
    if split_dirs:
        return "yolo_nolabelmap", split_dirs

    raise RuntimeError(
        f"В {dataset_root} не нашёл ни YOLO data.yaml, ни COCO .json, ни VOC .xml, "
        "ни пар images/+labels/. Проверьте структуру датасета вручную (os.walk)."
    )


def read_yolo_native_names(yaml_path):
    content = yaml.safe_load(open(yaml_path, encoding="utf-8"))
    names = content["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=lambda x: int(x))]
    return names


print("Хелперы загружены")

## 3. Конвертеры в YOLO-формат (все используют `link_or_copy`, ничего не дублируют на диске)

In [ ]:
def build_filename_index(root):
    """Строим индекс 'имя файла -> путь' ОДИН РАЗ, вместо пересканирования
    всего дерева датасета на каждое отдельное изображение (то, из-за чего
    конвертация больших датасетов вроде MOCS могла зависать без единого
    признака прогресса)."""
    index = {}
    for p in Path(root).rglob("*"):
        if p.is_file():
            index.setdefault(p.name, p)
    return index


def convert_coco_to_yolo(coco_json_path, images_search_root, out_images_dir, out_labels_dir,
                          progress_every=2000):
    out_images_dir = Path(out_images_dir); out_labels_dir = Path(out_labels_dir)
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    coco = json.load(open(coco_json_path, encoding="utf-8"))
    categories = sorted(coco["categories"], key=lambda c: c["id"])
    cat_id_to_idx = {c["id"]: i for i, c in enumerate(categories)}
    class_names = [c["name"] for c in categories]

    images_by_id = {img["id"]: img for img in coco["images"]}
    anns_by_image = {}
    for ann in coco["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    print(f"  строю индекс файлов в {images_search_root}...", flush=True)
    filename_index = build_filename_index(images_search_root)
    print(f"  индекс готов: {len(filename_index)} файлов", flush=True)

    n = 0
    total = len(images_by_id)
    for image_id, img_info in images_by_id.items():
        file_name = img_info["file_name"]
        w, h = img_info["width"], img_info["height"]

        src = Path(images_search_root) / file_name
        if not src.exists():
            src = filename_index.get(Path(file_name).name)
            if src is None:
                continue

        dst = out_images_dir / Path(file_name).name
        link_or_copy(src, dst)

        lines = []
        for ann in anns_by_image.get(image_id, []):
            x, y, bw, bh = ann["bbox"]
            cx, cy, nw, nh = (x + bw / 2) / w, (y + bh / 2) / h, bw / w, bh / h
            lines.append(f"{cat_id_to_idx[ann['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        (out_labels_dir / (Path(file_name).stem + ".txt")).write_text("\n".join(lines))
        n += 1
        if n % progress_every == 0:
            print(f"    ...{n}/{total}", flush=True)

    print(f"  готово: {n}/{total}", flush=True)
    return class_names, n


def convert_voc_to_yolo(xml_files, images_search_root, out_images_dir, out_labels_dir,
                         class_names=None, progress_every=2000):
    """Если class_names не переданы - собираем список классов автоматически по всем <name> в xml
    (важно: для НОВЫХ датасетов НЕЛЬЗЯ подставлять MOCS_CLASSES, у них свои классы!)."""
    out_images_dir = Path(out_images_dir); out_labels_dir = Path(out_labels_dir)
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    if class_names is None:
        discovered = set()
        for xml_path in xml_files:
            root = ET.parse(xml_path).getroot()
            for obj in root.findall("object"):
                discovered.add(obj.find("name").text)
        class_names = sorted(discovered)

    class_to_idx = {name: i for i, name in enumerate(class_names)}

    print(f"  строю индекс файлов в {images_search_root}...", flush=True)
    filename_index = build_filename_index(images_search_root)
    print(f"  индекс готов: {len(filename_index)} файлов", flush=True)

    n = 0
    total = len(xml_files)
    for xml_path in xml_files:
        root = ET.parse(xml_path).getroot()
        filename = root.find("filename").text
        size = root.find("size")
        w, h = int(size.find("width").text), int(size.find("height").text)

        src = filename_index.get(Path(filename).name)
        if src is None:
            continue
        dst = out_images_dir / Path(filename).name
        link_or_copy(src, dst)

        lines = []
        for obj in root.findall("object"):
            name = obj.find("name").text
            if name not in class_to_idx:
                continue
            bbox = obj.find("bndbox")
            xmin, ymin = float(bbox.find("xmin").text), float(bbox.find("ymin").text)
            xmax, ymax = float(bbox.find("xmax").text), float(bbox.find("ymax").text)
            cx, cy = ((xmin + xmax) / 2) / w, ((ymin + ymax) / 2) / h
            nw, nh = (xmax - xmin) / w, (ymax - ymin) / h
            lines.append(f"{class_to_idx[name]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        (out_labels_dir / (Path(filename).stem + ".txt")).write_text("\n".join(lines))
        n += 1
        if n % progress_every == 0:
            print(f"    ...{n}/{total}", flush=True)

    print(f"  готово: {n}/{total}", flush=True)
    return class_names, n


def convert_yolo_native(yaml_path, out_images_dir, out_labels_dir):
    """Датасет уже в YOLO-формате со своим data.yaml (частый случай для Roboflow-экспортов) -
    просто симлинкуем картинки и лейблы как есть, без перезаписи координат."""
    ds_root = Path(yaml_path).parent
    content = yaml.safe_load(open(yaml_path, encoding="utf-8"))
    class_names = read_yolo_native_names(yaml_path)

    out_images_dir = Path(out_images_dir); out_labels_dir = Path(out_labels_dir)
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    src_images_dirs = []
    for split_key in ("train", "val"):
        val = content.get(split_key)
        if val:
            p = (ds_root / val) if not os.path.isabs(val) else Path(val)
            if p.is_dir():
                src_images_dirs.append(p)
    if not src_images_dirs:
        src_images_dirs = list(ds_root.rglob("images"))

    n = 0
    for img_dir in src_images_dirs:
        lbl_dir = Path(str(img_dir).replace("images", "labels"))
        for img_path in img_dir.glob("*"):
            if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            link_or_copy(img_path, out_images_dir / img_path.name)
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                link_or_copy(lbl_path, out_labels_dir / lbl_path.name)
            else:
                (out_labels_dir / (img_path.stem + ".txt")).write_text("")
            n += 1

    return class_names, n


def convert_yolo_splits(split_dirs, out_images_dir, out_labels_dir, progress_every=2000):
    """Датасет в YOLO-формате БЕЗ data.yaml/classes.txt - есть только пары images/+labels/
    (train/, valid/, test/...). Сливаем все сплиты в единый пул - дальше по нему
    проедется наш общий split_train_val 90/10. Имена классов сюда явно передаются
    отдельно через class_names_override в prepare_dataset, угадать их неоткуда."""
    out_images_dir = Path(out_images_dir); out_labels_dir = Path(out_labels_dir)
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)
    total = sum(1 for d in split_dirs for p in (d / "images").glob("*")
                if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    print(f"  всего изображений по всем сплитам: {total}", flush=True)
    n = 0
    for split_dir in split_dirs:
        img_dir, lbl_dir = split_dir / "images", split_dir / "labels"
        for img_path in img_dir.glob("*"):
            if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            dst_img = out_images_dir / f"{split_dir.name}__{img_path.name}"
            link_or_copy(img_path, dst_img)
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            dst_lbl = out_labels_dir / (dst_img.stem + ".txt")
            if lbl_path.exists():
                link_or_copy(lbl_path, dst_lbl)
            else:
                dst_lbl.write_text("")
            n += 1
            if n % progress_every == 0:
                print(f"    ...{n}/{total}", flush=True)
    print(f"  готово: {n}/{total}", flush=True)

    return n


def print_class_distribution(labels_dir, class_names):
    """Печатает, сколько раз встречается каждый индекс - свериться глазами, что
    предполагаемый порядок классов реально совпадает с индексами внутри .txt."""
    from collections import Counter
    counts = Counter()
    for txt_path in Path(labels_dir).glob("*.txt"):
        for line in txt_path.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    for i, name in enumerate(class_names):
        print(f"  {i:2d}: {name:<25s} {counts.get(i, 0)} инстансов")


def remap_and_filter_labels(src_labels_dir, dst_labels_dir, src_names, name_map):
    """Переписывает YOLO .txt: старые индексы -> канонические 13 классов (по ИМЕНИ класса).
    Классы без соответствия в name_map выбрасываются построчно (картинка не удаляется целиком)."""
    src_labels_dir = Path(src_labels_dir); dst_labels_dir = Path(dst_labels_dir)
    dst_labels_dir.mkdir(parents=True, exist_ok=True)

    idx_remap, dropped_classes = {}, []
    for i, name in enumerate(src_names):
        canon_name = name_map.get(name.strip().lower())
        if canon_name is None:
            dropped_classes.append(name)
            continue
        idx_remap[i] = CANON_IDX[canon_name]

    n_files = n_kept = n_dropped = 0
    for txt_path in src_labels_dir.glob("*.txt"):
        out_lines = []
        for line in txt_path.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            old_idx = int(parts[0])
            if old_idx in idx_remap:
                parts[0] = str(idx_remap[old_idx])
                out_lines.append(" ".join(parts))
                n_kept += 1
            else:
                n_dropped += 1
        (dst_labels_dir / txt_path.name).write_text("\n".join(out_lines))
        n_files += 1

    return {"files": n_files, "boxes_kept": n_kept, "boxes_dropped": n_dropped, "dropped_classes": dropped_classes}


def split_train_val(images_dir, labels_dir, out_root, val_fraction=0.1, seed=42,
                     move=False, progress_every=2000):
    """move=True - перемещаем файлы вместо копирования (мгновенно, не зависит от
    symlink-ов) - используйте только когда images_dir - временная папка, которую
    и так удалите сразу после (как raw_dir в prepare_dataset). Для MOCS-пересплита
    в существующей на месте папке move НЕ используйте - там источник и назначение
    пересекаются."""
    images_dir = Path(images_dir); labels_dir = Path(labels_dir); out_root = Path(out_root)
    images = [p for p in images_dir.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
    random.seed(seed)
    random.shuffle(images)
    split_idx = int(len(images) * (1 - val_fraction))
    splits = {"train": images[:split_idx], "val": images[split_idx:]}

    total = len(images)
    done = 0
    for split_name, imgs in splits.items():
        img_out = out_root / f"images/{split_name}"
        lbl_out = out_root / f"labels/{split_name}"
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        for img_path in imgs:
            lbl_path = labels_dir / (img_path.stem + ".txt")
            dst_img, dst_lbl = img_out / img_path.name, lbl_out / lbl_path.name
            if move:
                if img_path.resolve() != dst_img.resolve():
                    shutil.move(str(img_path), str(dst_img))
                if lbl_path.exists() and lbl_path.resolve() != dst_lbl.resolve():
                    shutil.move(str(lbl_path), str(dst_lbl))
            else:
                link_or_copy(img_path, dst_img)
                if lbl_path.exists():
                    link_or_copy(lbl_path, dst_lbl)
            done += 1
            if done % progress_every == 0:
                print(f"    ...{done}/{total}", flush=True)
    print(f"  train/val готово: {done}/{total}", flush=True)
    return {k: len(v) for k, v in splits.items()}


def prepare_dataset(dataset_root, name_map, out_dir, val_fraction=0.1, class_names_override=None):
    """Единая точка входа: скачанный датасет произвольного формата -> папка *_yolo/
    с картинками (симлинки) и лейблами, уже переписанными в канонические 13 классов.

    class_names_override - передайте явно, если у датасета нет data.yaml/classes.txt
    (формат 'yolo_nolabelmap') и вы знаете порядок классов из другого источника
    (например, со страницы датасета на Kaggle)."""
    out_dir = Path(out_dir)
    if (out_dir / "images/train").exists() and any((out_dir / "images/train").iterdir()):
        print(f"  {out_dir} уже подготовлен, пропускаю (удалите папку вручную, если нужно пересобрать)")
        return out_dir

    raw_dir = out_dir / "_raw"
    fmt, info = detect_annotation_format(dataset_root)
    print(f"  формат: {fmt}")

    if fmt == "yolo_nolabelmap":
        if class_names_override is None:
            raise RuntimeError(
                "Датасет в YOLO-формате без data.yaml/classes.txt - "
                "нужно передать class_names_override с известным порядком классов."
            )
        n = convert_yolo_splits(info, raw_dir / "images", raw_dir / "labels")
        class_names = class_names_override
        print("  распределение классов ДО ремаппинга (сверьте глазами!):")
        print_class_distribution(raw_dir / "labels", class_names)
    elif fmt == "yolo":
        class_names, n = convert_yolo_native(info, raw_dir / "images", raw_dir / "labels")
    elif fmt == "coco":
        train_json = info[0]
        class_names, n = convert_coco_to_yolo(train_json, dataset_root, raw_dir / "images", raw_dir / "labels")
    else:  # voc
        class_names, n = convert_voc_to_yolo(info, dataset_root, raw_dir / "images", raw_dir / "labels")

    print(f"  найдено классов: {class_names}")
    print(f"  сконвертировано изображений: {n}")

    remap_stats = remap_and_filter_labels(raw_dir / "labels", out_dir / "_remapped_labels", class_names, name_map)
    print(f"  ремаппинг: {remap_stats}")
    if remap_stats["dropped_classes"]:
        print(f"  !! классы без маппинга (проигнорированы): {remap_stats['dropped_classes']}")

    split_stats = split_train_val(raw_dir / "images", out_dir / "_remapped_labels", out_dir, val_fraction, move=True)
    print(f"  train/val: {split_stats}")
    shutil.rmtree(raw_dir, ignore_errors=True)
    shutil.rmtree(out_dir / "_remapped_labels", ignore_errors=True)
    return out_dir


print("Конвертеры загружены")

## 4. MOCS — только если ещё не сконвертирован

**Не пересобираем MOCS, если `mocs_yolo/` уже существует с прошлого запуска** — это самый большой
датасет, пересобирать его лишний раз незачем (и лишний расход диска, и время).

In [ ]:
# Официальные названия классов в MOCS отличаются написанием от CANONICAL_CLASSES
# ("Static crane" против "tower_crane" и т.п.), хотя по смыслу это то же самое.
# Ремаппим ПО ИМЕНИ, как и остальные 4 датасета - так надёжнее, чем полагаться
# на то, что порядок категорий в исходном json всегда будет идентичен нашему.
MOCS_NAME_MAP = {
    "worker": "worker",
    "static crane": "tower_crane",
    "hanging head": "hanging_hook",
    "crane": "vehicle_crane",
    "roller": "roller",
    "bulldozer": "bulldozer",
    "excavator": "excavator",
    "truck": "truck",
    "loader": "loader",
    "pump truck": "pump_truck",
    "concrete mixer": "concrete_mixer",
    "pile driving": "pile_driver",
    "other vehicle": "other_vehicle",
}

if (MOCS_YOLO_DIR / "images/train").exists():
    print(f"MOCS уже сконвертирован в {MOCS_YOLO_DIR}, пропускаем")
else:
    if RUNNING_ON_KAGGLE:
        MOCS_DATASET_ROOT = "/kaggle/input/mocs-dataset"  # смонтирован, диск не расходует
    else:
        import kagglehub
        MOCS_DATASET_ROOT = kagglehub.dataset_download("xiaopan9802/mocs-dataset")

    print("MOCS путь:", MOCS_DATASET_ROOT)
    fmt, info = detect_annotation_format(MOCS_DATASET_ROOT)
    print("формат MOCS:", fmt)

    raw_dir = MOCS_YOLO_DIR / "_raw"

    if fmt == "coco":
        train_json = next((f for f in info if "train" in str(f).lower()), info[0])
        val_json = next((f for f in info if "val" in str(f).lower()), None)
        class_names, n_train = convert_coco_to_yolo(
            train_json, MOCS_DATASET_ROOT, raw_dir / "images/train", raw_dir / "labels/train"
        )
        print(f"MOCS train: {n_train} изображений")
        if val_json:
            convert_coco_to_yolo(val_json, MOCS_DATASET_ROOT, raw_dir / "images/val", raw_dir / "labels/val")
        else:
            print("Отдельного val.json нет - сплит 90/10 сделаем ниже из train")
    elif fmt == "voc":
        class_names, n_all = convert_voc_to_yolo(
            info, MOCS_DATASET_ROOT, raw_dir / "images/train", raw_dir / "labels/train", class_names=None
        )
    else:  # уже в YOLO
        class_names, n = convert_yolo_native(info, raw_dir / "images/train", raw_dir / "labels/train")

    print("Классы в исходной MOCS-разметке:", class_names)

    for split in ("train", "val"):
        src_labels = raw_dir / "labels" / split
        if not src_labels.exists():
            continue
        stats = remap_and_filter_labels(src_labels, MOCS_YOLO_DIR / "labels" / split, class_names, MOCS_NAME_MAP)
        print(f"  {split}: {stats}")
        if stats["dropped_classes"]:
            raise RuntimeError(
                f"В MOCS есть классы без соответствия в MOCS_NAME_MAP: {stats['dropped_classes']}. "
                "Для официального датасета таких быть не должно - проверьте название вручную."
            )
        img_out = MOCS_YOLO_DIR / "images" / split
        img_out.mkdir(parents=True, exist_ok=True)
        for img_path in (raw_dir / "images" / split).glob("*"):
            link_or_copy(img_path, img_out / img_path.name)

    val_dir = MOCS_YOLO_DIR / "images/val"
    if not val_dir.exists() or not any(val_dir.iterdir()):
        print("val отдельно не было - делаю сплит 90/10 из train")
        split_train_val(MOCS_YOLO_DIR / "images/train", MOCS_YOLO_DIR / "labels/train", MOCS_YOLO_DIR, val_fraction=0.1)

    shutil.rmtree(raw_dir, ignore_errors=True)
    class_names = CANONICAL_CLASSES
    print("MOCS приведён к канонической таксономии:", class_names)

## 5. Карты соответствия классов для новых датасетов

Ключ — имя класса в исходном датасете (в нижнем регистре, как оно там называется), значение — один
из `CANONICAL_CLASSES`. Классы, которых нет в словаре (например, PPE-классы у roboflow-датасета —
`helmet`, `vest`, `no-helmet` и т.п., они не про технику), автоматически выбрасываются в `remap_and_filter_labels`.

In [ ]:
NAME_MAP_XYZYXZZXY = {
    "dump truck": "truck",
    "excavator": "excavator",
    "motor grader": "other_vehicle",
    "roller": "roller",
    "crane manipulator": "vehicle_crane",
    "gazelle": "other_vehicle",
    "forklift standart": "other_vehicle",
    "bucket loader big": "loader",
    "mixer": "concrete_mixer",
    "tanker": "other_vehicle",
    "bulldozer": "bulldozer",
    "cleaning equipment": "other_vehicle",
    "truck": "truck",
    "trailer": "other_vehicle",
    "forklift giraffe": "other_vehicle",
    "bucket loader standart": "loader",
    "autocran": "vehicle_crane",
}

NAME_MAP_KARTAVIYCHERT = {
    "crane": "vehicle_crane",
    "excavator": "excavator",
    "truck": "truck",
    "tractor": "other_vehicle",
    "other": "other_vehicle",
}

NAME_MAP_DATACLUSTERLABS = {
    "truck": "truck",
    "tractor": "other_vehicle",
}

NAME_MAP_ROBOFLOW_JEJUNG = {
    "crane": "vehicle_crane",
    "pump-truck": "pump_truck",
    "roller": "roller",
    "excavator-crane": "excavator",   # составной класс, ближе всего к excavator - стоит перепроверить глазами
    "concrete truck": "concrete_mixer",
    "aerial work truck": "other_vehicle",
    "engineer vehicle": "other_vehicle",
    "generator": "other_vehicle",
    "trailer": "other_vehicle",
    "shovel": "excavator",
    "smalldigger": "excavator",
    # helmet, vest, cone, no-helmet, no-vest, safety shoes, no safety shoes, PA-helmet, PA-vest,
    # PanAsia engineer, HondaSUV, motorcycle, sub-helmet - намеренно не мапим, это не техника
}

print("Карты соответствия готовы")

## 6. Скачивание и подготовка новых датасетов

Каждый датасет получает свою отдельную папку в `finetune_data/` — они не сливаются друг с другом
и тем более не сливаются с `mocs_yolo/`. `prepare_dataset()` сама определяет формат разметки,
конвертирует, ремаппит классы и делает train/val split (90/10).

Заполните способ скачивания под то, что у вас реально работает (`kagglehub` для kaggle-датасетов,
Roboflow SDK для roboflow — там нужен свой API-ключ, впишите его в `ROBOFLOW_API_KEY`).

In [ ]:
import kagglehub

# У этого датасета нет data.yaml/classes.txt в раздаче - порядок классов
# берём со страницы Kaggle Data Explorer (0: Dump truck, 1: Excavator, ...).
# После конвертации ОБЯЗАТЕЛЬНО проверьте вывод print_class_distribution -
# если распределение выглядит неправдоподобно (например, у редкого класса
# тысячи инстансов), значит реальный порядок в файлах другой.
XYZYXZZXY_CLASS_NAMES = [
    "Dump truck", "Excavator", "Motor grader", "Roller", "Crane manipulator", "Gazelle",
    "Forklift Standart", "Bucket loader Big", "Mixer", "Tanker", "Bulldozer",
    "Cleaning equipment", "Truck", "Trailer", "Forklift Giraffe", "Bucket loader Standart", "Autocran",
]

datasets_to_prepare = [
    ("xyzyxzzxy/construction-equipment", NAME_MAP_XYZYXZZXY, "extra_xyzyxzzxy", XYZYXZZXY_CLASS_NAMES),
    ("kartaviychert/arh-df", NAME_MAP_KARTAVIYCHERT, "extra_kartaviychert", None),
    ("dataclusterlabs/construction-vehicle-images", NAME_MAP_DATACLUSTERLABS, "extra_dataclusterlabs", None),
]

prepared_dirs = []
for kaggle_slug, name_map, out_name, class_override in datasets_to_prepare:
    print(f"\n=== {kaggle_slug} ===")
    raw_root = kagglehub.dataset_download(kaggle_slug)
    out_dir = prepare_dataset(raw_root, name_map, WORK_ROOT / out_name, class_names_override=class_override)
    prepared_dirs.append(out_dir)

In [ ]:
import kagglehub

# У этого датасета нет data.yaml/classes.txt в раздаче - порядок классов
# берём со страницы Kaggle Data Explorer (0: Dump truck, 1: Excavator, ...).
# После конвертации ОБЯЗАТЕЛЬНО проверьте вывод print_class_distribution -
# если распределение выглядит неправдоподобно (например, у редкого класса
# тысячи инстансов), значит реальный порядок в файлах другой.
XYZYXZZXY_CLASS_NAMES = [
    "Dump truck", "Excavator", "Motor grader", "Roller", "Crane manipulator", "Gazelle",
    "Forklift Standart", "Bucket loader Big", "Mixer", "Tanker", "Bulldozer",
    "Cleaning equipment", "Truck", "Trailer", "Forklift Giraffe", "Bucket loader Standart", "Autocran",
]

datasets_to_prepare = [
    ("xyzyxzzxy/construction-equipment", NAME_MAP_XYZYXZZXY, "extra_xyzyxzzxy", XYZYXZZXY_CLASS_NAMES),
    ("kartaviychert/arh-df", NAME_MAP_KARTAVIYCHERT, "extra_kartaviychert", None),
    ("dataclusterlabs/construction-vehicle-images", NAME_MAP_DATACLUSTERLABS, "extra_dataclusterlabs", None),
]

prepared_dirs = []
for kaggle_slug, name_map, out_name, class_override in datasets_to_prepare:
    print(f"\n=== {kaggle_slug} ===")
    out_dir_check = WORK_ROOT / out_name
    if (out_dir_check / "images/train").exists() and any((out_dir_check / "images/train").iterdir()):
        print(f"  {out_dir_check} уже подготовлен, пропускаю (без скачивания)")
        prepared_dirs.append(out_dir_check)
        continue
    raw_root = kagglehub.dataset_download(kaggle_slug)
    out_dir = prepare_dataset(raw_root, name_map, WORK_ROOT / out_name, class_names_override=class_override)
    prepared_dirs.append(out_dir)

In [ ]:
# Roboflow-датасет: своя SDK, свой API-ключ (получить на universe.roboflow.com/settings/api).
# Если ключа нет под рукой - пропустите эту ячейку, остальной пайплайн от неё не зависит.

ROBOFLOW_API_KEY = "ВАШ_КЛЮЧ_ТУТ"

try:
    from roboflow import Roboflow

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("jejung").project("construction-machinery-tbosw")
    version = project.version(1)
    roboflow_raw_dir = version.download("yolov8").location

    out_dir = prepare_dataset(roboflow_raw_dir, NAME_MAP_ROBOFLOW_JEJUNG, WORK_ROOT / "extra_roboflow_jejung")
    prepared_dirs.append(out_dir)
except Exception as e:
    print(f"Roboflow-датасет пропущен: {e}")
    print("Остальной пайплайн продолжит работать и без него.")

## 7. Итоговый data.yaml — список путей, без физического слияния

MOCS занимает своё место на диске один раз, новые датасеты — тоже по одному разу каждый.
Ничего никуда дополнительно не копируется.

In [ ]:
train_paths = [str((MOCS_YOLO_DIR / "images/train").resolve())]
val_paths = [str((MOCS_YOLO_DIR / "images/val").resolve())]

for d in prepared_dirs:
    train_paths.append(str((d / "images/train").resolve()))
    val_paths.append(str((d / "images/val").resolve()))

merged_yaml_lines = ["train:"]
merged_yaml_lines += [f"  - {p}" for p in train_paths]
merged_yaml_lines.append("val:")
merged_yaml_lines += [f"  - {p}" for p in val_paths]
merged_yaml_lines.append("names:")
merged_yaml_lines += [f"  {i}: {name}" for i, name in enumerate(CANONICAL_CLASSES)]

MERGED_DATA_YAML = WORK_ROOT / "merged_data.yaml"
MERGED_DATA_YAML.write_text("\n".join(merged_yaml_lines))
print(MERGED_DATA_YAML.read_text())

## 8. Проверка перед запуском обучения

In [ ]:
for p in train_paths:
    n_img = len(list(Path(p).glob("*")))
    lbl_dir = Path(p.replace("images", "labels"))
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"{p}\n  изображений: {n_img}, файлов разметки: {n_lbl}")

import yaml as _yaml
with open(MERGED_DATA_YAML) as f:
    parsed = _yaml.safe_load(f)
assert parsed["names"] == {i: name for i, name in enumerate(CANONICAL_CLASSES)}
print("\ndata.yaml валиден, порядок классов совпадает с CANONICAL_CLASSES")

## 9. Дообучение — две фазы, без `resume`

**Фаза 1** — backbone заморожен (`freeze`), несколько эпох, чтобы новые данные не "смыли" уже
выученные признаки на сильных классах (Excavator, Worker).
**Фаза 2** — полная разморозка, низкий learning rate, `cos_lr`, early stopping по `patience`.

`cache=False` в обеих фазах — не кэшируем изображения (датасет с MOCS внутри слишком большой
для RAM- или disk-кэша).

In [ ]:
from ultralytics import YOLO

assert Path(PREV_BEST_WEIGHTS).exists(), f"Не найдены веса {PREV_BEST_WEIGHTS} - проверьте путь"

model = YOLO(PREV_BEST_WEIGHTS)  # веса как стартовая точка, НЕ resume

model.train(
    data=str(MERGED_DATA_YAML),
    epochs=5,
    imgsz=640,
    batch=16,
    freeze=10,           # число замороженных слоёв backbone (для yolov8m обычно 10-12)
    lr0=0.001,
    cache=False,
    project="runs_finetune",
    name="phase1_frozen",
    exist_ok=True,
)

In [ ]:
from pathlib import Path
from ultralytics import YOLO

checks = {
    "mocs_yolo": Path(r"C:\Users\edwar\PycharmProjects\JupyterProject\cv\mocs_yolo"),
    "extra_xyzyxzzxy": WORK_ROOT / "extra_xyzyxzzxy",
    "extra_kartaviychert": WORK_ROOT / "extra_kartaviychert",
    "extra_dataclusterlabs": WORK_ROOT / "extra_dataclusterlabs",
}

for name, base in checks.items():
    for split in ("train", "val"):
        img_dir = base / "images" / split
        lbl_dir = base / "labels" / split
        n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
        n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
        print(f"{name:22s} {split:5s} изображений: {n_img:6d}  разметки: {n_lbl:6d}")

In [ ]:
model_phase2 = YOLO(r"C:\Users\edwar\PycharmProjects\JupyterProject\cv\runs\detect\runs_finetune/phase1_frozen/weights/last.pt")

results = model_phase2.train(
    data=str(MERGED_DATA_YAML),
    epochs=20,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.0005,
    cos_lr=True,
    patience=8,
    cache=False,
    project="runs_finetune",
    name="phase2_full_v2",
    exist_ok=True,
)

best_weights_path = model_phase2.trainer.best
print("Итоговые лучшие веса:", best_weights_path)

In [ ]:
# Ручной запуск инференса без обучения: путь к уже готовым весам phase2_full
best_weights_path = r"C:\Users\edwar\PycharmProjects\JupyterProject\cv\runs\detect\runs_finetune\phase2_full\weights\best.pt"
print("Используем веса:", best_weights_path)

## 10. Инференс и трекинг на видео

In [ ]:
from datetime import datetime, timedelta
import cv2
import numpy as np

trained_model = YOLO(best_weights_path)


def run_video_tracking(video_path, model, video_start_time, conf=0.35, tracker="bytetrack.yaml",
                        sample_every_n_frames=1, motion_patch_size=48):
    """Детекция + трекинг по видео. Для каждой детекции дополнительно считает motion_energy -
    меру внутрикадрового движения ВНУТРИ рамки объекта (сравнение уменьшенного grayscale-патча
    текущего бокса с патчем того же track_id в предыдущем сохранённом кадре). Это нужно, чтобы
    отличать "работает на месте" (ковш/стрела движутся, а база стоит) от простоя - одно смещение
    центра рамки такое не ловит."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.release()

    prev_patches = {}  # track_id -> последний grayscale-патч
    detections = []

    stream = model.track(source=video_path, conf=conf, tracker=tracker, stream=True, verbose=False)

    for frame_idx, result in enumerate(stream):
        if frame_idx % sample_every_n_frames != 0:
            continue
        timestamp = video_start_time + timedelta(seconds=frame_idx / fps)
        if result.boxes is None:
            continue

        frame = result.orig_img  # BGR numpy-массив текущего кадра, отдаёт сама Ultralytics
        for box in result.boxes:
            class_name = model.names[int(box.cls[0])]
            track_id = int(box.id[0]) if box.id is not None else None
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            motion_energy = None
            if track_id is not None:
                xi1, yi1, xi2, yi2 = int(max(x1, 0)), int(max(y1, 0)), int(x2), int(y2)
                crop = frame[yi1:yi2, xi1:xi2]
                if crop.size > 0:
                    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
                    patch = cv2.resize(gray, (motion_patch_size, motion_patch_size))
                    if track_id in prev_patches:
                        diff = np.abs(patch.astype(np.int16) - prev_patches[track_id].astype(np.int16))
                        motion_energy = float(diff.mean())
                    prev_patches[track_id] = patch

            detections.append({
                "timestamp": timestamp,
                "frame_idx": frame_idx,
                "track_id": track_id,
                "class_name": class_name,
                "confidence": float(box.conf[0]),
                "bbox": (x1, y1, x2, y2),
                "motion_energy": motion_energy,  # None на первом появлении track_id, дальше - число
            })

    return detections


VIDEO_PATH = "your_video.mp4"
detections = run_video_tracking(VIDEO_PATH, trained_model, datetime.now())
print("Всего детекций:", len(detections))

## 10b. Экспорт копии видео с отрисованными рамками

Прогоняет видео через `model.track()` ещё раз и для каждого кадра рисует рамки, класс,
track_id и confidence через встроенный `result.plot()`, после чего пишет кадр в выходной
видеофайл через `cv2.VideoWriter`. Результат — mp4-копия исходного видео с наложенной
разметкой в реальном времени (как её "видела" бы модель на каждом кадре).

In [ ]:
def export_annotated_video(video_path, model, output_path, conf=0.35, tracker="bytetrack.yaml",
                            progress_every=100, font_size=10, line_width=5):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    # mp4v совместим с большинством плееров (VLC и т.п.); если видео не откроется в браузере,
    # замените fourcc на "avc1" (нужен ffmpeg с поддержкой H.264 в сборке OpenCV).
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    stream = model.track(source=video_path, conf=conf, tracker=tracker, stream=True, verbose=False)
    for frame_idx, result in enumerate(stream):
        annotated_frame = result.plot(font_size=font_size, line_width=line_width)  # BGR-кадр с рамками/классами/id/confidence
        writer.write(annotated_frame)
        if progress_every and frame_idx % progress_every == 0:
            print(f"кадр {frame_idx}/{total_frames}")

    writer.release()
    print("Готово:", output_path)


OUTPUT_VIDEO_PATH = "your_video_annotated.mp4"
export_annotated_video(VIDEO_PATH, trained_model, OUTPUT_VIDEO_PATH)

## 11. Подсчёт уникальной техники по track_id

In [ ]:
import pandas as pd
from collections import defaultdict


def count_equipment(detections):
    tracks_by_class = defaultdict(set)
    for det in detections:
        if det["track_id"] is not None:
            tracks_by_class[det["class_name"]].add(det["track_id"])
    return {cls: len(ids) for cls, ids in tracks_by_class.items()}


equipment_counts = count_equipment(detections)
for cls, count in equipment_counts.items():
    print(f"  {cls}: {count}")

pd.DataFrame(equipment_counts.items(), columns=["class_name", "count"])

## 11b. Активность и простой техники

Одного смещения центра рамки недостаточно: экскаватор или кран может "работать", стоя на месте
(двигается ковш/стрела), а не только переезжая. Поэтому активность считается по ДВУМ сигналам:
смещение центра рамки (переехала) ИЛИ высокая motion_energy внутри рамки (работает на месте).
Активным считается временное окно, где сработал хотя бы один из сигналов.

In [ ]:
def compute_activity(detections, movement_threshold_px=15, motion_energy_threshold=6.0, window_seconds=60):
    """Группирует детекции по track_id и временным окнам, классифицирует каждое окно
    как "active" или "idle". movement_threshold_px подберите под реальное разрешение видео и
    масштаб сцены (крупный план vs общий вид площадки будут давать разный масштаб смещения
    в пикселях на одно и то же физическое перемещение) - начните с дефолта и скорректируйте
    по факту, глядя на пару размеченных примеров."""
    from collections import defaultdict

    by_track = defaultdict(list)
    for d in detections:
        if d["track_id"] is None:
            continue
        x1, y1, x2, y2 = d["bbox"]
        by_track[d["track_id"]].append({**d, "cx": (x1 + x2) / 2, "cy": (y1 + y2) / 2})

    activity_log = []
    for track_id, points in by_track.items():
        points.sort(key=lambda p: p["timestamp"])
        class_name = points[0]["class_name"]

        window_start_idx = 0
        for i in range(1, len(points)):
            elapsed = (points[i]["timestamp"] - points[window_start_idx]["timestamp"]).total_seconds()
            is_last = i == len(points) - 1
            if elapsed >= window_seconds or is_last:
                seg = points[window_start_idx:i + 1]
                dx = seg[-1]["cx"] - seg[0]["cx"]
                dy = seg[-1]["cy"] - seg[0]["cy"]
                displacement = (dx ** 2 + dy ** 2) ** 0.5

                motion_values = [p["motion_energy"] for p in seg if p["motion_energy"] is not None]
                max_motion = max(motion_values) if motion_values else 0.0

                moved = displacement > movement_threshold_px
                worked_in_place = max_motion > motion_energy_threshold
                status = "active" if (moved or worked_in_place) else "idle"

                activity_log.append({
                    "track_id": track_id, "class_name": class_name,
                    "window_start": seg[0]["timestamp"], "window_end": seg[-1]["timestamp"],
                    "displacement_px": round(displacement, 1), "max_motion_energy": round(max_motion, 2),
                    "status": status,
                    "reason": "moved" if moved else ("worked_in_place" if worked_in_place else "idle"),
                })
                window_start_idx = i
    return activity_log


def summarize_activity(activity_log):
    """Сводка по каждой единице техники: сколько окон активна/простаивает, доля простоя -
    это то, что пойдёт в отчёт как основной вывод модуля."""
    from collections import defaultdict

    stats_by_track = defaultdict(lambda: {"active": 0, "idle": 0, "class_name": None})
    for entry in activity_log:
        s = stats_by_track[entry["track_id"]]
        s["class_name"] = entry["class_name"]
        s[entry["status"]] += 1

    rows = []
    for track_id, s in stats_by_track.items():
        total = s["active"] + s["idle"]
        rows.append({
            "track_id": track_id, "class_name": s["class_name"],
            "active_windows": s["active"], "idle_windows": s["idle"],
            "idle_share": round(s["idle"] / total, 2) if total else 0,
        })
    return pd.DataFrame(rows).sort_values("idle_share", ascending=False)


activity_log = compute_activity(detections, window_seconds=60)
activity_summary = summarize_activity(activity_log)
activity_summary

## 11c. События прибытия/исчезновения техники

Строится журнал событий: для каждого отслеживаемого объекта (`track_id`) берётся
первое появление в кадре (`arrival`) и последнее (`departure`). Класс техники берётся
как самый частый `class_name` среди детекций этого объекта.

Трекер (ByteTrack) может потерять объект при долгом перекрытии и выдать ему при повторном
появлении новый `track_id` — тогда вместо одного объекта получились бы "лишние"
departure + arrival. `merge_track_splits` склеивает такие разорванные треки: если трек
того же класса начинается вскоре после (`max_gap_seconds`) и рядом в кадре
(`max_center_dist_px`) с концом другого, они считаются одним объектом.

In [ ]:
from typing import Literal
from pydantic import BaseModel


# Классы объектов детектора MOCS (Moving Objects on Construction Sites).
EquipmentClass = Literal[
    "worker",
    "tower_crane",
    "hanging_hook",
    "vehicle_crane",
    "roller",
    "bulldozer",
    "excavator",
    "truck",
    "loader",
    "pump_truck",
    "concrete_mixer",
    "pile_driver",
    "other_vehicle",
]


class EquipmentEvent(BaseModel):
    track_id: int
    equipment_class: EquipmentClass
    event: Literal["arrival", "departure"]
    at: datetime


def _bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def _dist(p1, p2):
    return ((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2) ** 0.5


def _build_track_summaries(df):
    summaries = []
    for track_id, group in df.groupby("track_id"):
        group = group.sort_values("timestamp")
        first, last = group.iloc[0], group.iloc[-1]
        summaries.append({
            "track_id": int(track_id),
            "equipment_class": group["class_name"].mode().iloc[0],
            "start_time": first["timestamp"],
            "end_time": last["timestamp"],
            "first_bbox_center": _bbox_center(first["bbox"]),
            "last_bbox_center": _bbox_center(last["bbox"]),
        })
    return summaries


def merge_track_splits(track_summaries, max_gap_seconds=3.0, max_center_dist_px=80.0):
    """Объединяет разорванные треки: если байт-трекер после долгого перекрытия выдал
    объекту новый track_id, то трек того же класса, начавшийся вскоре (<= max_gap_seconds)
    и рядом в кадре (<= max_center_dist_px от последнего bbox) с концом предыдущего трека,
    считается продолжением того же объекта. Возвращает {track_id: id_объединённой_группы}."""
    parent = {t["track_id"]: t["track_id"] for t in track_summaries}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    ordered = sorted(track_summaries, key=lambda t: t["start_time"])
    open_chains = []  # хвосты ещё не завершённых цепочек треков

    for track in ordered:
        best_chain, best_gap = None, None
        for chain in open_chains:
            if chain["equipment_class"] != track["equipment_class"]:
                continue
            gap = (track["start_time"] - chain["end_time"]).total_seconds()
            if gap < 0 or gap > max_gap_seconds:
                continue
            if _dist(chain["last_bbox_center"], track["first_bbox_center"]) > max_center_dist_px:
                continue
            if best_gap is None or gap < best_gap:
                best_gap, best_chain = gap, chain

        if best_chain is not None:
            union(best_chain["track_id"], track["track_id"])
            open_chains.remove(best_chain)

        open_chains.append({
            "track_id": track["track_id"],
            "end_time": track["end_time"],
            "last_bbox_center": track["last_bbox_center"],
            "equipment_class": track["equipment_class"],
        })

    return {t["track_id"]: find(t["track_id"]) for t in track_summaries}


def build_equipment_events(detections, max_gap_seconds=3.0, max_center_dist_px=80.0):
    """Строит список событий arrival/departure по объектам, предварительно склеив разорванные
    треки через merge_track_splits. track_id события - наименьший исходный track_id в группе."""
    if not detections:
        return []

    df = pd.DataFrame(detections)
    df = df[df["track_id"].notna()].copy()
    df["track_id"] = df["track_id"].astype(int)

    track_summaries = _build_track_summaries(df)
    merge_map = merge_track_splits(
        track_summaries, max_gap_seconds=max_gap_seconds, max_center_dist_px=max_center_dist_px,
    )
    df["group_id"] = df["track_id"].map(merge_map)

    events = []
    for group_id, group in df.groupby("group_id"):
        equipment_class = group["class_name"].mode().iloc[0]
        representative_track_id = int(group["track_id"].min())
        events.append(EquipmentEvent(
            track_id=representative_track_id, equipment_class=equipment_class,
            event="arrival", at=group["timestamp"].min(),
        ))
        events.append(EquipmentEvent(
            track_id=representative_track_id, equipment_class=equipment_class,
            event="departure", at=group["timestamp"].max(),
        ))

    events.sort(key=lambda e: e.at)
    return events


def format_elapsed(at, start_time):
    total_seconds = int((at - start_time).total_seconds())
    minutes, seconds = divmod(total_seconds, 60)
    return f"{minutes:02d}:{seconds:02d}"


equipment_events = build_equipment_events(detections)
video_start = min(d["timestamp"] for d in detections) if detections else None

for ev in equipment_events:
    verb = "прибыл" if ev.event == "arrival" else "исчез"
    print(f"{ev.equipment_class} #{ev.track_id} {verb} на {format_elapsed(ev.at, video_start)}")

events_path = "equipment_events.json"
with open(events_path, "w", encoding="utf-8") as f:
    json.dump([e.model_dump(mode="json") for e in equipment_events], f, ensure_ascii=False, indent=2)
print("Сохранено:", events_path)

## 12. Визуализация детекций на кадре

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def show_frame_with_detections(video_path, frame_idx, detections):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        print("Не удалось прочитать кадр", frame_idx)
        return

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(frame_rgb)

    for det in [d for d in detections if d["frame_idx"] == frame_idx]:
        x1, y1, x2, y2 = det["bbox"]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f"{det['class_name']} #{det['track_id']}", color="red", fontsize=10, weight="bold")

    ax.axis("off")
    plt.show()


if detections:
    mid_frame = detections[len(detections) // 2]["frame_idx"]
    show_frame_with_detections(VIDEO_PATH, mid_frame, detections)

## 13. Графики динамики обучения (фаза 2)

In [ ]:
csv_path = Path("runs_finetune/phase2_full/results.csv")
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Fine-tune (фаза 2) — динамика обучения", fontsize=16, fontweight="bold")

epoch = df["epoch"] + 1

axes[0, 0].plot(epoch, df["train/box_loss"], label="box_loss")
axes[0, 0].plot(epoch, df["train/cls_loss"], label="cls_loss")
axes[0, 0].plot(epoch, df["train/dfl_loss"], label="dfl_loss")
axes[0, 0].set_title("Ошибки на train")
axes[0, 0].set_xlabel("Эпоха"); axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(alpha=0.3); axes[0, 0].legend()

for col in ["val/box_loss", "val/cls_loss", "val/dfl_loss"]:
    if col in df.columns:
        axes[0, 1].plot(epoch, df[col], label=col.replace("val/", ""))
axes[0, 1].set_title("Ошибки на validation")
axes[0, 1].set_xlabel("Эпоха"); axes[0, 1].set_ylabel("Loss")
axes[0, 1].grid(alpha=0.3); axes[0, 1].legend()

axes[1, 0].plot(epoch, df["metrics/precision(B)"], label="Precision")
axes[1, 0].plot(epoch, df["metrics/recall(B)"], label="Recall")
axes[1, 0].set_title("Precision и Recall")
axes[1, 0].set_xlabel("Эпоха"); axes[1, 0].set_ylabel("Значение")
axes[1, 0].set_ylim(0, 1); axes[1, 0].grid(alpha=0.3); axes[1, 0].legend()

axes[1, 1].plot(epoch, df["metrics/mAP50(B)"], label="mAP50")
axes[1, 1].plot(epoch, df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[1, 1].set_title("Качество детекции")
axes[1, 1].set_xlabel("Эпоха"); axes[1, 1].set_ylabel("mAP")
axes[1, 1].set_ylim(0, 1); axes[1, 1].grid(alpha=0.3); axes[1, 1].legend()

plt.tight_layout()
output_path = csv_path.parent / "metrics_over_epochs.png"
plt.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"График сохранён: {output_path}")

## 13. Графики динамики обучения (фаза 2)

In [ ]:
csv_path = Path("runs_finetune/phase2_full/results.csv")
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Fine-tune (фаза 2) — динамика обучения", fontsize=16, fontweight="bold")

epoch = df["epoch"] + 1

axes[0, 0].plot(epoch, df["train/box_loss"], label="box_loss")
axes[0, 0].plot(epoch, df["train/cls_loss"], label="cls_loss")
axes[0, 0].plot(epoch, df["train/dfl_loss"], label="dfl_loss")
axes[0, 0].set_title("Ошибки на train")
axes[0, 0].set_xlabel("Эпоха"); axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(alpha=0.3); axes[0, 0].legend()

for col in ["val/box_loss", "val/cls_loss", "val/dfl_loss"]:
    if col in df.columns:
        axes[0, 1].plot(epoch, df[col], label=col.replace("val/", ""))
axes[0, 1].set_title("Ошибки на validation")
axes[0, 1].set_xlabel("Эпоха"); axes[0, 1].set_ylabel("Loss")
axes[0, 1].grid(alpha=0.3); axes[0, 1].legend()

axes[1, 0].plot(epoch, df["metrics/precision(B)"], label="Precision")
axes[1, 0].plot(epoch, df["metrics/recall(B)"], label="Recall")
axes[1, 0].set_title("Precision и Recall")
axes[1, 0].set_xlabel("Эпоха"); axes[1, 0].set_ylabel("Значение")
axes[1, 0].set_ylim(0, 1); axes[1, 0].grid(alpha=0.3); axes[1, 0].legend()

axes[1, 1].plot(epoch, df["metrics/mAP50(B)"], label="mAP50")
axes[1, 1].plot(epoch, df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[1, 1].set_title("Качество детекции")
axes[1, 1].set_xlabel("Эпоха"); axes[1, 1].set_ylabel("mAP")
axes[1, 1].set_ylim(0, 1); axes[1, 1].grid(alpha=0.3); axes[1, 1].legend()

plt.tight_layout()
output_path = csv_path.parent / "metrics_over_epochs.png"
plt.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"График сохранён: {output_path}")